# Data preparation and splitting

The default starts from all 1,600 factoid questions in data/training13b.json and randomly splits sorted IDs **without stratification**, using seed 3407. The initial split contains 1,440 train and 160 dev questions. Dev remains unfiltered: **121 snippet-matching and 39 non-matching questions**. Only the training split is filtered, retaining **1,137 questions / 1,363 alias rows**. The 95 official test questions stay separate.

split_order=split_first is the default. The supported_dev_legacy preset (split_order=filter_first) reproduces the earlier 1,132-train/126-dev protocol. Changing training filter mode with split_first never changes dev membership. All question types here means supported/unsupported factoids; list, yes/no and summary questions are outside this experiment.

Change dev_ratio, seed, filter_mode (normalized, literal, none), and alias_mode (per_alias, first, all). Normalized matching changes punctuation/case/Greek and letter-digit forms; literal matching preserves case and punctuation. Neither verifies semantic support. The all-alias output requires a multi-answer prompt rather than the single-answer instruction.

The output train_prepared.json is for SFT; train_questions.json has one record per question for candidate generation or further SFT/DPO splitting; dev_prepared.json retains all accepted aliases for scoring. The full_resources preset includes questions without snippet matches. reserve_dpo_questions creates a separate SFT/DPO reservation, not a replacement dev split.

For prepared data, split question records before expanding aliases. occurrence_evidence takes prepared source and exports evidence-plus-answer/control records without API calls. judge_evidence uses semantic annotation; export_judged_evidence requires a complete judgment cache as annotation_dir. To continue annotation into a fresh run, supply the prior cache as annotation_dir. Only historical flags actually present in the supplied source are carried forward.

Dev evidence references have a separate pipeline: dev_occurrence, dev_judge, then dev_export, using prepared dev source and question-level train_source to verify separation. Occurrence-only references are diagnostics; the evidence SFT trainer requires the completed semantic v2 export from dev_export. Annotation batches can continue using the prior annotation_dir cache.

Execution is **off by default**. Preview needs only Python and the repository. Select a kernel with the project's runtime dependencies before executing. Each method uses that kernel's Python in a separate worker process.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/notebook_workflows").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_workflows import describe, plan, execute
from src.notebook_workflows.presets import PRESETS, configuration
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)


## Choose a preset

- **snippet_supported**: snippet_split
- **supported_dev_legacy**: snippet_split
- **literal_supported**: snippet_split
- **full_resources**: snippet_split
- **one_supported_alias**: snippet_split
- **prepared_train_dev**: prepared_split
- **reserve_dpo_questions**: sft_dpo_split
- **occurrence_evidence**: occurrence_export
- **judge_evidence**: evidence_annotation
- **export_judged_evidence**: evidence_annotation
- **dev_occurrence**: dev_evidence
- **dev_judge**: dev_evidence
- **dev_export**: dev_evidence

Edit the configuration, then rerun the remaining cells. OVERRIDES uses the option names printed by describe(), with underscores. For CLI flags, True includes the named flag and False omits it. Missing inputs block execution.


In [ ]:
CATEGORY = 'data'
PRESET = 'snippet_supported'
OVERRIDES = {
    # Set input/model paths here using the fields printed below.
}
RUN = False
ALLOW_API = False

# Optional sequential comparisons: shared OVERRIDES plus per-variant changes.
VARIANTS = [{"label": "default", "overrides": {}}]

METHOD, PARAMETERS = configuration(CATEGORY, PRESET, OVERRIDES)
print(json.dumps({"method": METHOD, "parameters": PARAMETERS}, indent=2))


## Inspect and preview

Options are read from source without importing the training stack. Unknown options and invalid choices are rejected. Outputs go under Artifacts/notebook_runs and cannot overwrite earlier runs. Rerun this cell to allocate fresh plans.


In [ ]:
print(json.dumps(describe(METHOD), indent=2, default=str))
PLANS = []
for variant in VARIANTS:
    method, parameters = configuration(CATEGORY, PRESET, {**OVERRIDES, **variant["overrides"]})
    preview = plan(method, parameters, project_root=PROJECT_ROOT)
    PLANS.append(preview)
    print("\nVariant:", variant["label"])
    print("Method:", preview["method"])
    print("Output:", preview["run_dir"])
    print("Requires GPU:", preview["gpu"], "| Calls API:", preview["api"])
    print("Unresolved inputs:", preview["missing"] or "none")
    print(json.dumps(preview["parameters"], indent=2))


## Execute reviewed plans

Set RUN=True, then rerun configuration and preview. API methods also need ALLOW_API=True. Plans cannot be changed after preview. Each run retains its configuration, input hashes, status and log. A repeated execution of the same plan refuses to overwrite its directory; create a fresh plan instead.


In [ ]:
RESULTS = []
for preview in PLANS:
    if RUN:
        result = execute(preview, run=True, allow_api=ALLOW_API)
        RESULTS.append(str(result))
    else:
        print("Preview only:", preview["method"], "- set RUN=True to execute.")


## Inspect outputs

Use these explicit paths in the next notebook. Historical commands, plots and outputs are preserved in reproducibility/notebook_archive.zip; notebooks/MIGRATION.md maps old notebooks to these workflows.


In [ ]:
for run in RESULTS:
    path = Path(run)
    print("\nRun:", path)
    print((path / "status.json").read_text(encoding="utf-8"))
    print("Log:", path / "run.log")
    print("Outputs:", [p.name for p in path.iterdir()
                      if p.name not in {"run.log", "plan.json", "input_hashes.json"}])
